# Ateneo cleaning checks (task 6)

Checks for the Ateneo cleaning code in `src/transform/` (rules H in notebook 01). The pipeline runs that code through `clean_backfill.py`. This notebook only tests it and shows the results.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
RAW = PROJECT_ROOT / "data" / "raw"
STAGING = PROJECT_ROOT / "data" / "staging"

ateneo = pd.read_parquet(RAW / "ateneo_politicians_raw.parquet")
print(ateneo.shape)

(207599, 10)


## 1. Text repair, scanning errors, and suffixes

In [2]:
from src.transform.normalize import fix_mojibake, fix_ocr_digits, split_suffix

for col in ["first_name", "last_name", "party", "municipality"]:
    values = ateneo[col].dropna()
    print(col, "| values repaired:", (values.map(fix_mojibake) != values).sum())

towns = ateneo["municipality"].dropna().drop_duplicates()
repaired = towns.map(fix_mojibake)
pd.DataFrame({"before": towns, "after": repaired})[repaired != towns].head()

first_name | values repaired: 0
last_name | values repaired: 0
party | values repaired: 31
municipality | values repaired: 363


,before,after
1255,SOFRONIO ESPAÃ‘OLA,SOFRONIO ESPAÑOLA
3668,DUEÃ‘AS,DUEÑAS
7190,SERGIO OSMEÃ‘A SR.,SERGIO OSMEÑA SR.
7192,LOS BAÃ‘OS,LOS BAÑOS
8757,PARAÃ‘AQUE,PARAÑAQUE


In [3]:
for name in ["CARIN0", "FILOMEN0", "DOM1NADOR", "PLARIDEL 1V", "ALAN1", "VICTOR 10"]:
    print(f"{name:12} -> {fix_ocr_digits(name)}")

CARIN0       -> CARINO
FILOMEN0     -> FILOMENO
DOM1NADOR    -> DOMINADOR
PLARIDEL 1V  -> PLARIDEL IV
ALAN1        -> ALAN1
VICTOR 10    -> VICTOR 10


In [4]:
examples = ["JOSE JR.", "SALICJR.", "ADOLFOJR", "GREGORIO 111", "REYNOLD JR. II", "SERGIO ANTONIO V", "NASR", "LEVI"]
pd.DataFrame({
    "name": examples,
    "split": [split_suffix(n) for n in examples],
    "split, 2019 rule": [split_suffix(n, glued_ok=True) for n in examples],
})

,name,split,"split, 2019 rule"
0,JOSE JR.,"(JOSE, Jr.)","(JOSE, Jr.)"
1,SALICJR.,"(SALIC, Jr.)","(SALIC, Jr.)"
2,ADOLFOJR,"(ADOLFOJR, None)","(ADOLFO, Jr.)"
3,GREGORIO 111,"(GREGORIO, III)","(GREGORIO, III)"
4,REYNOLD JR. II,"(REYNOLD, Jr. II)","(REYNOLD, Jr. II)"
5,SERGIO ANTONIO V,"(SERGIO ANTONIO V, None)","(SERGIO ANTONIO V, None)"
6,NASR,"(NASR, None)","(NASR, None)"
7,LEVI,"(LEVI, None)","(LEVI, None)"


### Summary: text repair, scanning errors, and suffixes

- **Broken text:** 363 municipality values and 31 party values had a broken enye ("PARAÃ‘AQUE"). `fix_mojibake` repairs all of them. Names had none.
- **Scanning errors:** a 0 next to a letter becomes O, and a 1 right before a letter becomes I (CARIN0 → CARINO, PLARIDEL 1V → PLARIDEL IV). Digits that are not clearly letters stay as they are (ALAN1, VICTOR 10). Those rows get the `name_suspect` flag later.
- **Suffixes:** `split_suffix` moves JR., SR., II, III and so on into their own value. This includes "111" (→ III) and suffixes stuck to the name with a period (SALICJR.). Suffixes stuck to the name without a period (ADOLFOJR) are split only in 2019, the year spaces were lost. A final V stays, because it may be a middle initial.

## 2. First cleaning run (before the town review)

In [5]:
from src.transform.clean_backfill import clean_ateneo_politicians, known_town_pairs
from src.transform.mappings import ATENEO_PROVINCE_MAP

memberships = pd.read_parquet(STAGING / "hf_memberships_clean.parquet")
winners = pd.read_parquet(STAGING / "openhalalan_winners_clean.parquet")
known = known_town_pairs(memberships, winners)
at = clean_ateneo_politicians(ateneo, known)

print(at.shape, "| known towns:", len(known))
print("town_source:", at["town_source"].value_counts().to_dict())
print("town_verified:", at["town_verified"].value_counts(dropna=False).to_dict())

(207599, 24) | known towns: 1692
town_source: {'original': 172376, 'unfilled': 22032, 'not_applicable': 13191}
town_verified: {True: 170865, nan: 35223, False: 1511}


In [6]:
moved = at["province_std"] != at["province"].replace(ATENEO_PROVINCE_MAP)
print("rows moved to the right NCR district:", moved.sum())
at[moved].groupby(["year", "province", "town_std", "province_std"]).size()

rows moved to the right NCR district: 58


year  province               town_std  province_std       
1988  NCR - FOURTH DISTRICT  PASIG     NCR SECOND DISTRICT    14
1992  NCR - FOURTH DISTRICT  PASIG     NCR SECOND DISTRICT    14
      NCR - THIRD DISTRICT   SAN JUAN  NCR SECOND DISTRICT     1
1995  NCR - SECOND DISTRICT  NAVOTAS   NCR THIRD DISTRICT      1
      NCR - THIRD DISTRICT   SAN JUAN  NCR SECOND DISTRICT     1
1998  NCR - FOURTH DISTRICT  PASIG     NCR SECOND DISTRICT    14
      NCR - THIRD DISTRICT   SAN JUAN  NCR SECOND DISTRICT     1
2019  NCR - THIRD DISTRICT   PATEROS   NCR FOURTH DISTRICT    12
dtype: int64

In [7]:
print("suffixes:", at["name_suffix"].value_counts().to_dict())
print("scanning errors:", at["name_suspect"].sum(), "| duplicate rows:", at["is_duplicate"].sum(), "| fat dynasty rows:", at["is_fat_dynasty"].sum())
at.loc[at["year"].eq(2019) & at["name_suffix"].notna(), ["first_name", "first_name_std", "name_suffix"]].head()

suffixes: {'Jr.': 2350, 'Sr.': 414, 'III': 223, 'II': 144, 'IV': 40, 'VI': 9, 'VII': 1, 'Jr. II': 1, 'Jr. IV': 1, 'VIII': 1}
scanning errors: 91 | duplicate rows: 13 | fat dynasty rows: 52932


,first_name,first_name_std,name_suffix
171819,ABBASJR.,ABBAS,Jr.
171866,JAIMESR.,JAIME,Sr.
171913,LACSJR.,LACS,Jr.
171936,AMERJR.,AMER,Jr.
172133,SALICJR.,SALIC,Jr.


In [8]:
unverified = at[at["town_verified"].eq(False)]
print(unverified["town_std"].nunique(), "town names in", len(unverified), "rows are not in HF or OpenHalalan")
unverified["year"].value_counts().sort_index()

114 town names in 1511 rows are not in HF or OpenHalalan


year
1988    776
1992    228
1995    166
1998    341
Name: count, dtype: int64

### Summary: first cleaning run

- **Rows:** all 207,599 rows are kept, with 14 new columns (row id, standard places, name parts, and flags).
- **Towns:** 172,376 town-level rows have a town. 22,032 have none: 15,842 in 2013 and 6,180 in 2016 (the gaps HF already filled, which section 5 copies over), plus 10 rows in 1988 written "---". The other 13,191 rows are provincial or House positions, which have no town.
- **NCR districts:** 58 rows filed under the wrong Metro Manila district moved to their city's district.
- **Names:** 3,184 suffixes moved to `name_suffix`, 91 names with scanning errors flagged, and 13 duplicate rows flagged.
- **Unknown towns:** 1,511 rows (114 names, all from 1988 to 1998) use a town name that neither HF nor OpenHalalan has. Section 3 reviews them.

## 3. Town name review

A town name counts as verified when HF or OpenHalalan records the same town in the same province group. For each unknown name, `suggest_town_fixes` gives two clues: a known town of the same province with almost the same spelling, and the known town where most of the same surnames held office within the next 10 years. We accept a suggestion when it makes sense, and write down the reason whenever we decide differently. HF's spelling stays the standard, so existing outputs keep their names.

### 3a. OpenHalalan names that HF never uses

In [10]:
from src.transform.mappings import PROVINCE_GROUP
from src.transform.normalize import name_key
from src.utils.town_review import suggest_town_fixes

persons = pd.read_parquet(STAGING / "hf_persons_clean.parquet")
town_positions = ["MAYOR", "VICE MAYOR", "COUNCILOR"]

hf = memberships[memberships["position"].isin(town_positions) & memberships["locality_source"].eq("original")]
hf = hf.merge(persons[["id", "last_key"]], left_on="person_id", right_on="id")
hf = hf.rename(columns={"locality_std": "town_std"})
hf["province_group"] = hf["province_std"].replace(PROVINCE_GROUP)

oh = winners[winners["position"].isin(town_positions) & ~winners["town_shift_suspect"] & winners["town_std"].notna()].copy()
oh["province_group"] = oh["province_std"].replace(PROVINCE_GROUP)
oh["last_key"] = oh["last_name"].map(name_key, na_action="ignore")

hf_towns = set(zip(hf["province_group"], hf["town_std"]))
oh_only = oh[[pair not in hf_towns for pair in zip(oh["province_group"], oh["town_std"])]]
review_oh = suggest_town_fixes(oh_only, hf)
print(len(review_oh), "OpenHalalan town names that HF never uses, in", review_oh["rows"].sum(), "rows")
review_oh

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
56 OpenHalalan town names that HF never uses, in 1372 rows


,province_group,town_std,rows,years,spelling,shared_town,shared,of,suggestion
0,ABRA,SAL LAPADAN,10,[2001],SALLAPADAN,SALLAPADAN,6,10,SALLAPADAN
1,BATANGAS,MATAASNAKAHOY,10,[2025],MATAAS NA KAHOY,NaN,0,10,MATAAS NA KAHOY
2,BATANGAS,STO TOMAS,20,"[2022, 2025]",SANTO TOMAS,NaN,0,10,SANTO TOMAS
3,BENGUET,LA TRINIDAD CAPITAL,19,"[2004, 2007]",NaN,LA TRINIDAD,9,9,LA TRINIDAD
4,BOHOL,GETAFE,20,"[2022, 2025]",JETAFE,NaN,0,9,JETAFE
5,BULACAN,BALIWAG,10,[2025],BALIUAG,NaN,0,8,BALIUAG
6,CEBU,PINAMUNGAJAN,30,"[2001, 2022, 2025]",PINAMUNGAHAN,PINAMUNGAHAN,9,10,PINAMUNGAHAN
7,COTABATO,KADAYANGAN,10,[2025],NaN,NaN,0,10,NaN
8,COTABATO,KAPALAWAN,10,[2025],NaN,NaN,0,10,NaN
9,COTABATO,LIGAWASAN,10,[2025],NaN,NaN,0,9,NaN


In [11]:
decisions = {
    "TUGUNAN": None,                           # a new town in the Special Geographic Area (2025), not TULUNAN
    "KALOOCAN": "CALOOCAN",                    # official spelling
    "KALOOKAN": "CALOOCAN",                    # HF's other spelling of the same city
    "AMAI MANABILANG": "BUMBARAN",             # the town's new name; HF uses the old one
    "LEON T POSTIGO": "BACUNGAN",              # the town's new name; HF uses the old one
    "SHARIFF AGUAK CAPITAL": "SHARIFF AGUAK",  # same name plus "capital"
}
fixes = dict(zip(review_oh["town_std"], review_oh["suggestion"]))
fixes.update(decisions)
for town, std in sorted(fixes.items()):
    if pd.notna(std):
        print(f'    "{town}": "{std}",')
print("no change (new towns):", sorted(t for t, s in fixes.items() if pd.isna(s)))

    "AMAI MANABILANG": "BUMBARAN",
    "AMAI MANABILANG BUMBARAN": "BUMBARAN",
    "BACOLOD KALAWI BACOLOD GRANDE": "BACOLOD KALAWI",
    "BACUNGAN LEON T POSTIGO": "BACUNGAN",
    "BALIWAG": "BALIUAG",
    "BANAY BANAY": "BANAYBANAY",
    "BAYAWAN CITY TULONG": "BAYAWAN",
    "BINDOY PAYABON": "BINDOY",
    "BROOKE S POINT": "BROOKES POINT",
    "BULALACAO SAN PEDRO": "BULALACAO",
    "D V CHIONGBIAN D M MARCOS": "DON VICTORIANO CHIONGBIAN",
    "DON V CHIONGBIAN": "DON VICTORIANO CHIONGBIAN",
    "DUMAGUETE CITY CAPITAL": "DUMAGUETE",
    "GABALDON BITULOK & SABANI": "GABALDON",
    "GABALDON BITULOK AND SABANI": "GABALDON",
    "GENERAL SANTOS CITY DADIANGAS": "GENERAL SANTOS",
    "GETAFE": "JETAFE",
    "HADJI PANGLIMA TAHIL MARUNGGAS": "HADJI PANGLIMA TAHIL",
    "JAVIER BUGHO": "JAVIER",
    "JOLO CAPITAL": "JOLO",
    "KABUGAO CAPITAL": "KABUGAO",
    "KALOOCAN": "CALOOCAN",
    "KALOOKAN": "CALOOCAN",
    "LA TRINIDAD CAPITAL": "LA TRINIDAD",
    "LAGAWE BURNAY": "LAGAWE",
   

### Summary: OpenHalalan names

- 56 OpenHalalan town names (1,372 rows) were never used by HF. 48 are the same towns written differently: an old name or "capital" attached (SALCEDO BAUGEN, JOLO CAPITAL), a spelling (POZORRUBIO, T BOLI), or a newer official name (GETAFE → JETAFE, AMAI MANABILANG → BUMBARAN). They were added to `TOWN_MAP` with HF's spelling as the standard. Caloocan's three spellings (CALOOCAN, KALOOCAN, KALOOKAN) are now one.
- The other 8 are new towns first elected in 2025 (Special Geographic Area), so they stay as they are. The code suggested TULUNAN for TUGUNAN, and we rejected it.
- Effect on the existing outputs: 1,435 OpenHalalan rows now use HF's spelling, 42 HF rows changed from KALOOKAN to CALOOCAN, and 4 more HF rows got a town (SASMUAN, BAYAWAN, MLANG). Persons, the ID map, PSA and the roster did not change.

### 3b. Ateneo names that no other source has

In [12]:
import importlib
import src.transform.mappings
import src.transform.clean_backfill
importlib.reload(src.transform.mappings)        # the notebook still holds the old maps in memory,
importlib.reload(src.transform.clean_backfill)  # so reload both, mappings first
from src.transform.clean_backfill import clean_ateneo_politicians, known_town_pairs

memberships = pd.read_parquet(STAGING / "hf_memberships_clean.parquet")
winners = pd.read_parquet(STAGING / "openhalalan_winners_clean.parquet")
at = clean_ateneo_politicians(ateneo, known_town_pairs(memberships, winners))
unverified = at[at["town_verified"].eq(False)]
review_at = suggest_town_fixes(unverified, at[at["town_verified"].eq(True)])
print(len(review_at), "Ateneo town names in", review_at["rows"].sum(), "rows | with a suggestion:", review_at["suggestion"].notna().sum())
pd.set_option("display.max_rows", 200)
review_at[["province_group", "town_std", "rows", "years", "spelling", "shared_town", "shared", "of", "suggestion"]]

115 Ateneo town names in 1511 rows | with a suggestion: 102


,province_group,town_std,rows,years,spelling,shared_town,shared,of,suggestion
0,ABRA,BAAY LICUAN,20,"[1992, 1998]",NaN,BUCLOC,3,10,BUCLOC
1,ABRA,LICUAN BAY,10,[1988],LICUAN BAAY,LICUAN BAAY,2,10,LICUAN BAAY
2,BATANGAS,SAN PACUAL,10,[1998],SAN PASCUAL,SAN PASCUAL,10,10,SAN PASCUAL
3,BOHOL,ANTEQUERRA,10,[1992],ANTEQUERA,ANTEQUERA,7,8,ANTEQUERA
4,BOHOL,CATIGBI AN,10,[1992],CATIGBIAN,CATIGBIAN,7,10,CATIGBIAN
5,BOHOL,PRES CP GARCIA,20,"[1995, 1998]",NaN,PRESIDENT CARLOS P GARCIA,6,10,PRESIDENT CARLOS P GARCIA
6,BOHOL,PRES GARCIA,10,[1992],NaN,PRESIDENT CARLOS P GARCIA,5,10,PRESIDENT CARLOS P GARCIA
7,BOHOL,PRESIDENTE CARLOS P GARCIA,10,[1988],PRESIDENT CARLOS P GARCIA,MABINI,2,10,PRESIDENT CARLOS P GARCIA
8,BUKIDNON,IMPASUGONG,10,[1998],IMPASUG ONG,IMPASUG ONG,5,10,IMPASUG ONG
9,BUKIDNON,MALAYBALAYY,10,[1988],MALAYBALAY,MALAYBALAY,4,10,MALAYBALAY


In [13]:
decisions = {
    ("ABRA", "BAAY LICUAN"): "LICUAN BAAY",                # words swapped; BUCLOC is listed separately in those years
    ("CAVITE", "MENDEZ NUNEZ"): "MENDEZ",                  # former name
    ("DAVAO DEL NORTE", "BABAK"): None,                    # a real town until 1998 (historical)
    ("DAVAO DEL NORTE", "KAPUT IAN"): "KAPUTIAN",          # spelling of a historical town
    ("ILOCOS NORTE", "ESPIRITU"): "BANNA",                 # former name, written "ESPIRITU (BANNA)"
    ("LANAO DEL SUR", "BACOLOD GRANDE"): "BACOLOD KALAWI", # former name
    ("MAGUINDANAO", "DINAIG"): "DATU ODIN SINSUAT",        # former name
    ("MAGUINDANAO", "MAGANOY"): "SHARIFF AGUAK",           # former name
    ("NUEVA ECIJA", "MUNOZ"): "SCIENCE CITY OF MUNOZ",     # became a city in 2000
}
fixes = dict(zip(zip(review_at["province_group"], review_at["town_std"]), review_at["suggestion"]))
fixes.update(decisions)
for (group, town), std in sorted(fixes.items()):
    if pd.notna(std):
        print(f'    ("{group}", "{town}"): "{std}",')
print("no fix:", sorted(key for key, std in fixes.items() if pd.isna(std)))

    ("ABRA", "BAAY LICUAN"): "LICUAN BAAY",
    ("ABRA", "LICUAN BAY"): "LICUAN BAAY",
    ("BATANGAS", "SAN PACUAL"): "SAN PASCUAL",
    ("BOHOL", "ANTEQUERRA"): "ANTEQUERA",
    ("BOHOL", "CATIGBI AN"): "CATIGBIAN",
    ("BOHOL", "PRES CP GARCIA"): "PRESIDENT CARLOS P GARCIA",
    ("BOHOL", "PRES GARCIA"): "PRESIDENT CARLOS P GARCIA",
    ("BOHOL", "PRESIDENTE CARLOS P GARCIA"): "PRESIDENT CARLOS P GARCIA",
    ("BUKIDNON", "IMPASUGONG"): "IMPASUG ONG",
    ("BUKIDNON", "MALAYBALAYY"): "MALAYBALAY",
    ("BULACAN", "BUSANTO S"): "BUSTOS",
    ("CAGAYAN", "LALLO"): "LAL LO",
    ("CAMARINES SUR", "GARCHITORINA"): "GARCHITORENA",
    ("CAPIZ", "MAAYON"): "MA AYON",
    ("CAPIZ", "SAPIAN"): "SAPI AN",
    ("CAVITE", "GEN E AGUINALDO"): "GENERAL EMILIO AGUINALDO",
    ("CAVITE", "GEN M ALVAREZ"): "GEN MARIANO ALVAREZ",
    ("CAVITE", "GEN TRIAS"): "GENERAL TRIAS",
    ("CAVITE", "GENERAL MARIANO ALVAREZ"): "GEN MARIANO ALVAREZ",
    ("CAVITE", "MENDEZ NUNEZ"): "MENDEZ",
    ("CAVITE", "N

In [14]:
importlib.reload(src.transform.mappings)
importlib.reload(src.transform.clean_backfill)
from src.transform.clean_backfill import clean_ateneo_politicians, known_town_pairs

at = clean_ateneo_politicians(ateneo, known_town_pairs(memberships, winners))
print("town_verified:", at["town_verified"].value_counts(dropna=False).to_dict())
at[at["town_verified"].eq(False)].groupby(["year", "province", "municipality", "town_std"]).size()

town_verified: {True: 172344, nan: 35223, False: 32}


year  province       municipality                 town_std          
1988  CAMARINES SUR  SAN AGUSTIN                  SAN AGUSTIN           10
      MAGUINDANAO    SULTAN SA MAROMATA           SULTAN SA MAROMATA    10
1995  MAGUINDANAO    MARAWI CITY (LANAO DEL SUR)  MARAWI                12
dtype: int64

### Summary: Ateneo names

- 115 Ateneo town names (1,511 rows, 1988 to 1998) were in neither HF nor OpenHalalan. 108 now map to a known town in `PROVINCE_TOWN_MAP`. 100 come from the code's suggestions: mostly spelling slips (TAGIG → TAGUIG), and 1988 scanning errors confirmed by the officials' surnames (MAUTA → MALITA, 8 of 10 surnames shared). 8 were decided by hand: former names (MUNOZ → SCIENCE CITY OF MUNOZ, ESPIRITU → BANNA), swapped words (BAAY LICUAN → LICUAN BAAY), and a spacing typo (KAPUT IAN). Two suggestions were rejected: BAAY LICUAN → BUCLOC and BABAK → BRAULIO E DUJALI.
- Four were real towns that later merged into cities (BABAK, KAPUTIAN, SAMAL, BACON). They keep their names and are listed in `HISTORICAL_TOWNS`.
- 32 rows remain unverified: SAN AGUSTIN (Camarines Sur, 1988) and SULTAN SA MAROMATA (Maguindanao, 1988), where no clue is strong enough, and Marawi, which was filed under Maguindanao in 1995 (fixed in section 4).

## 4. Town label check

A town name can be real and still be the wrong label for a group of rows. Families usually stay in the same town, so each town's officials should share some surnames with the same town in the next election. `label_suspects` flags town-years whose officials share at most 1 surname with their own town in the next election, but at least 4 with another town of the same province.

In [15]:
from src.utils.town_review import label_suspects

suspects = label_suspects(at[at["town_verified"].eq(True)])
print(len(suspects), "town-years whose officials look like another town's")
suspects

10 town-years whose officials look like another town's


,province_group,year,town_std,officials,shared_with_itself,next_year,best_other,shared_with_other
0,BULACAN,1998,BULACAN,8,0,2001,BOCAUE,6
1,DAVAO DEL NORTE,1992,BABAK,9,0,1995,BRAULIO E DUJALI,7
2,DAVAO DEL NORTE,1995,BRAULIO E DUJALI,10,0,1998,ISLAND GARDEN CITY OF SAMAL,4
3,ILOCOS NORTE,2016,BANNA ESPIRITU,9,0,2019,BANNA,7
4,LEYTE,1998,BILIRAN,9,0,2001,ALMERIA,6
5,NORTHERN SAMAR,1988,SAN ROQUE,10,0,1992,SAN VICENTE,4
6,NORTHERN SAMAR,1988,SAN VICENTE,10,0,1992,SAN ROQUE,5
7,NUEVA ECIJA,1988,LAUR,10,0,1992,LUPAO,5
8,PALAWAN,1988,ROXAS,10,0,1992,SAN VICENTE,4
9,QUEZON,1998,QUEZON,10,0,2001,POLILLO,7


In [16]:
counts = at[at["town_source"].eq("original")].groupby(["year", "province_std", "town_std"]).size()
counts[(counts > 16) & (counts.index.get_level_values("year") == 2001)]

year  province_std         town_std  
2001  BILIRAN              ALMERIA       20
      BULACAN              BOCAUE        20
      CEBU                 CEBU          18
      DAVAO DEL SUR        DAVAO         26
      LEYTE                LA PAZ        20
      NCR FIRST DISTRICT   MANILA        38
      NCR FOURTH DISTRICT  MAKATI        18
                           MUNTINLUPA    18
      NCR SECOND DISTRICT  QUEZON        26
      QUEZON               POLILLO       20
dtype: int64

In [17]:
from src.transform.clean_backfill import split_doubled_towns, standardize_winners

winners_std = standardize_winners(pd.read_csv(RAW / "openhalalan_winners.csv"))
at_split = split_doubled_towns(at, winners_std)
moved = at_split["town_std"].ne(at["town_std"]) & at["town_std"].notna()
print("rows given their own town:", moved.sum())
pd.DataFrame({"year": at.loc[moved, "year"], "province": at.loc[moved, "province_std"], "listed_as": at.loc[moved, "town_std"],
              "real_town": at_split.loc[moved, "town_std"]}).value_counts()

rows given their own town: 40


year  province  listed_as  real_town
2001  QUEZON    POLILLO    QUEZON       10
      LEYTE     LA PAZ     LEYTE        10
      BILIRAN   ALMERIA    BILIRAN      10
      BULACAN   BOCAUE     BULACAN      10
Name: count, dtype: int64

In [18]:
label_suspects(at_split[at_split["town_verified"].eq(True)])

,province_group,year,town_std,officials,shared_with_itself,next_year,best_other,shared_with_other
0,DAVAO DEL NORTE,1992,BABAK,9,0,1995,BRAULIO E DUJALI,7
1,DAVAO DEL NORTE,1995,BRAULIO E DUJALI,10,0,1998,ISLAND GARDEN CITY OF SAMAL,4
2,ILOCOS NORTE,2016,BANNA ESPIRITU,9,0,2019,BANNA,7
3,NORTHERN SAMAR,1988,SAN ROQUE,10,0,1992,SAN VICENTE,4
4,NORTHERN SAMAR,1988,SAN VICENTE,10,0,1992,SAN ROQUE,5
5,NUEVA ECIJA,1988,LAUR,10,0,1992,LUPAO,5
6,PALAWAN,1988,ROXAS,10,0,1992,SAN VICENTE,4


In [19]:
def surnames(df, group, year, town):
    rows = df[df["province_group"].eq(group) & df["year"].eq(year) & df["town_std"].eq(town)]
    return set(rows["last_key"])

sv_1988 = surnames(at_split, "PALAWAN", 1988, "SAN VICENTE")
for year in [1992, 1995]:
    print(f"Palawan SAN VICENTE 1988 shares with ROXAS {year}:", len(sv_1988 & surnames(at_split, "PALAWAN", year, "ROXAS")),
          f"| with SAN VICENTE {year}:", len(sv_1988 & surnames(at_split, "PALAWAN", year, "SAN VICENTE")))

at_split[at_split["town_std"].eq("MARAWI")].groupby(["year", "province"]).size().unstack(fill_value=0)

Palawan SAN VICENTE 1988 shares with ROXAS 1992: 3 | with SAN VICENTE 1992: 0
Palawan SAN VICENTE 1988 shares with ROXAS 1995: 4 | with SAN VICENTE 1995: 0


province,LANAO DEL SUR,MAGUINDANAO
year,,
1988,12,0
1992,12,0
1995,0,12
1998,12,0
2001,12,0
2004,12,0
2007,8,0
2010,12,0
2019,12,0


In [ ]:
importlib.reload(src.transform.mappings)
importlib.reload(src.transform.clean_backfill)
from src.transform.clean_backfill import clean_ateneo_politicians, known_town_pairs, split_doubled_towns, standardize_winners

memberships = pd.read_parquet(STAGING / "hf_memberships_clean.parquet")
winners = pd.read_parquet(STAGING / "openhalalan_winners_clean.parquet")
winners_std = standardize_winners(pd.read_csv(RAW / "openhalalan_winners.csv"))
at = split_doubled_towns(clean_ateneo_politicians(ateneo, known_town_pairs(memberships, winners)), winners_std)
print("town_verified:", at["town_verified"].value_counts(dropna=False).to_dict())
label_suspects(at[at["town_verified"].eq(True)])

### Summary: town label check

- The check flagged 10 town-years, and each one was explained and fixed:
  - **2001 doubled towns:** Ateneo put the officials of the four towns named like their province (BULACAN, BILIRAN, LEYTE, QUEZON) under the town listed before them (BOCAUE, ALMERIA, LA PAZ, POLILLO), which then showed 20 officials instead of 10. `split_doubled_towns` gives these 40 officials their own town back, using OpenHalalan's 2001 results for the same people.
  - **Filed under the wrong place in one election** (`ATENEO_PLACE_FIXES`, 7 entries): the 1988 labels of SAN ROQUE and SAN VICENTE (Northern Samar) and of ROXAS and SAN VICENTE (Palawan) are swapped. Ateneo's 1988 LAUR is LUPAO. Ateneo's 1995 BRAULIO E DUJALI is BABAK, since Braulio E. Dujali was only created in 1998. Marawi's 1995 officials were filed under Maguindanao.
  - **Old name attached:** BANNA ESPIRITU is BANNA (added to `TOWN_MAP`). This also changed 14 HF rows and 84 OpenHalalan rows.
- After the fixes, one flag remains, and it is expected: BABAK 1995 shares 4 surnames with Island Garden City of Samal 1998, because Babak merged into that city in 1998.
- Only 20 Ateneo rows now have a town that no other source confirms (SAN AGUSTIN and SULTAN SA MAROMATA, 1988).